In [27]:
#Make sure the correct packages are here, and that the embedding modules from the openAI github are here
!pip install pandas openai transformers plotly matplotlib scikit-learn torch torchvision scipy --quiet
!git clone https://github.com/openai/openai-cookbook.git

fatal: destination path 'openai-cookbook' already exists and is not an empty directory.


In [28]:
#append the openai cookbook to allow the notebook to import custom utility modules
import sys
sys.path.append("/content/openai-cookbook/examples")

In [29]:
#double check that embedding is here
!find /content -name "embeddings_utils.py"

/content/openai-cookbook/examples/utils/embeddings_utils.py


In [30]:
#openAI key I trust you guys
%run Marvel_API_KEY.ipynb

import os
os.environ["OPENAI_API_KEY"] = API_KEY

In [31]:
# start implementing the data
import json
import pandas as pd

In [33]:
# check actor data columns to make sure embeddings line up
df_actors = pd.read_json('actors.json')
print("ACTOR COLUMNS:", df_actors.columns)
print(df_actors.head(2))

# same for character data
df_chars = pd.read_json('Character-descriptions.json')
print("CHARACTER COLUMNS:", df_chars.columns)
print(df_chars.head(2))

ACTOR COLUMNS: Index(['actors'], dtype='object')
                                              actors
0  {'name': 'Robert Downey Jr.', 'description': '...
1  {'name': 'Chris Hemsworth', 'description': 'De...
CHARACTER COLUMNS: Index(['characters'], dtype='object')
                                          characters
0  {'name': 'Iron Man', 'traits': 'Genius billion...
1  {'name': 'Thor', 'traits': 'Leaping from the l...


In [34]:
import pandas as pd
import tiktoken #tiktoken converts text into tokens, as done in the example

from utils.embeddings_utils import get_embedding

# normalize the actors column which contains a list of dictionaries into line by line format with no blank space
df_actors_normalized = pd.json_normalize(df_actors['actors'])

df_actors_normalized['combined_text'] = (
    "Actor Name: " + df_actors_normalized['name'] + "; Bio: " + df_actors_normalized['description']
)

# embbed the actors using the openAI text embedding model
df_actors_normalized['embedding'] = df_actors_normalized.combined_text.apply(
    lambda x: get_embedding(x, model="text-embedding-3-small")
)

# normalize the characters column the same way
df_chars_normalized = pd.json_normalize(df_chars['characters'])

df_chars_normalized['combined_text'] = (
    "Character: " + df_chars_normalized['name'] +
    "; Traits: " + df_chars_normalized['traits'] +
    "; Powers: " + df_chars_normalized['powers']
)

# Embed the characters/tokens
df_chars_normalized['embedding'] = df_chars_normalized.combined_text.apply(
    lambda x: get_embedding(x, model="text-embedding-3-small")
)

In [35]:
from utils.embeddings_utils import cosine_similarity

def marvel_cast(character_name, df_actors, df_chars, n=3):
    # 1. find the character's embedding, and raise an error if not found in the character dataset
    try:
        character_embedding = df_chars.loc[df_chars['name'] == character_name, 'embedding'].values[0]
    except IndexError:
        return f"Character '{character_name}' not found in data."

    # compare this character embedding against ALL actor embeddings
    df_actors["similarity"] = df_actors.embedding.apply(
        lambda x: cosine_similarity(x, character_embedding)
    )

    # sort by highest similarity
    results = df_actors.sort_values("similarity", ascending=False).head(n)

    print(f"--- Casting Recommendations for {character_name} ---")
    for idx, row in results.iterrows():
        print(f"Actor: {row['name']}")
        print(f"Similarity Score: {row['similarity']:.4f}")
        print("-" * 10)

marvel_cast("Iron Man", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("Thor", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("James Howlett", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("Peter Parker", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("Victor Von Doom", df_actors_normalized, df_chars_normalized, n=3)


--- Casting Recommendations for Iron Man ---
Actor: Robert Downey Jr.
Similarity Score: 0.5939
----------
Actor: Chris Hemsworth
Similarity Score: 0.4201
----------
Actor: Mark Strong
Similarity Score: 0.3908
----------
--- Casting Recommendations for Thor ---
Actor: Chris Hemsworth
Similarity Score: 0.6138
----------
Actor: Chris Pratt
Similarity Score: 0.3976
----------
Actor: Samuel L. Jackson
Similarity Score: 0.3828
----------
--- Casting Recommendations for James Howlett ---
Actor: Hugh Jackman
Similarity Score: 0.5938
----------
Actor: Chris Hemsworth
Similarity Score: 0.3728
----------
Actor: Michael Fassbender
Similarity Score: 0.3714
----------
--- Casting Recommendations for Peter Parker ---
Actor: Tom Holland
Similarity Score: 0.5752
----------
Actor: Chris Pratt
Similarity Score: 0.3394
----------
Actor: Samuel L. Jackson
Similarity Score: 0.3383
----------
--- Casting Recommendations for Victor Von Doom ---
Actor: Chris Hemsworth
Similarity Score: 0.3490
----------
Actor:

# Part Two
Update the `marvel_cast` function by adding a step to normalize the similarity scores after calculating cosine similarity.

In [38]:
def normalize_similarity_score(scores):
    min_score = scores.min()
    max_score = scores.max()
    #function shown on website
    normalized_scores = (scores - min_score) / (max_score - min_score)
    return normalized_scores

# Print a message to confirm the function definition
print("Defined normalize_similarity_score function.")

Defined normalize_similarity_score function.


*Process:
 modify the `marvel_cast` function to include the normalization of similarity scores using the `normalize_similarity_score` function, and then re-run the casting recommendations.



In [39]:
from utils.embeddings_utils import cosine_similarity

def marvel_cast(character_name, df_actors, df_chars, n=3):
    # 1. find the character's embedding, and raise an error if not found in the character dataset
    try:
        character_embedding = df_chars.loc[df_chars['name'] == character_name, 'embedding'].values[0]
    except IndexError:
        return f"Character '{character_name}' not found in data."

    # compare this character embedding against ALL actor embeddings
    df_actors_copy = df_actors.copy() # Create a copy to avoid modifying the original DataFrame in place
    df_actors_copy["similarity"] = df_actors_copy.embedding.apply(
        lambda x: cosine_similarity(x, character_embedding)
    )

    # Normalize the similarity scores
    df_actors_copy["similarity"] = normalize_similarity_score(df_actors_copy["similarity"])

    # sort by highest similarity
    results = df_actors_copy.sort_values("similarity", ascending=False).head(n)

    print(f"--- Casting Recommendations for {character_name} ---")
    for idx, row in results.iterrows():
        print(f"Actor: {row['name']}")
        print(f"Similarity Score: {row['similarity']:.4f}")
        print("-" * 10)

marvel_cast("Iron Man", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("Thor", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("James Howlett", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("Peter Parker", df_actors_normalized, df_chars_normalized, n=3)
marvel_cast("Victor Von Doom", df_actors_normalized, df_chars_normalized, n=3)

--- Casting Recommendations for Iron Man ---
Actor: Robert Downey Jr.
Similarity Score: 1.0000
----------
Actor: Chris Hemsworth
Similarity Score: 0.6425
----------
Actor: Mark Strong
Similarity Score: 0.5822
----------
--- Casting Recommendations for Thor ---
Actor: Chris Hemsworth
Similarity Score: 1.0000
----------
Actor: Chris Pratt
Similarity Score: 0.5754
----------
Actor: Samuel L. Jackson
Similarity Score: 0.5464
----------
--- Casting Recommendations for James Howlett ---
Actor: Hugh Jackman
Similarity Score: 1.0000
----------
Actor: Chris Hemsworth
Similarity Score: 0.5490
----------
Actor: Michael Fassbender
Similarity Score: 0.5462
----------
--- Casting Recommendations for Peter Parker ---
Actor: Tom Holland
Similarity Score: 1.0000
----------
Actor: Chris Pratt
Similarity Score: 0.4732
----------
Actor: Samuel L. Jackson
Similarity Score: 0.4708
----------
--- Casting Recommendations for Victor Von Doom ---
Actor: Chris Hemsworth
Similarity Score: 1.0000
----------
Actor: